# 📋 claude.ai 프롬프트 / claude.ai Prompt

> **수업 활용법:** 아래 프롬프트를 [claude.ai](https://claude.ai)에 붙여넣으면 유사한 노트북을 생성할 수 있습니다.

---

### 🇰🇷 한국어 프롬프트
```
Anthropic Claude API의 Tool Use(도구 사용)를 활용한 AI 에이전트
Google Colab 노트북을 한국어로 작성해주세요.

0. 에이전트 핵심 용어 사전: 에이전트, 도구, 도구 스키마, 에이전트 루프,
   stop_reason, 오케스트레이터, ReAct 패턴을 표와 그림으로 설명
1. 에이전트 개념 설명: LLM + 도구 + 루프 구조
2. 파이썬 함수로 3가지 도구 정의:
   - get_current_time(): 현재 날짜와 시간 반환
   - calculate(expression): 수식 안전하게 계산 (eval 대신 ast 사용)
   - search_company_info(company_name): 딕셔너리에서 가상 회사 정보 반환
3. Anthropic API의 tools 파라미터로 도구를 Claude에게 전달
4. Tool Use 루프 구현 (stop_reason 기반, block.input 사용):
   while 루프: Claude 응답 → stop_reason 확인 → tool_use면 함수 실행
   → tool_result를 messages에 추가 → 다시 Claude 호출 → 최종 답변
5. 에이전트 루프 추적 출력
6. 3가지 테스트 시나리오
7. 미니 실습: 나만의 도구 추가하기
한국어 주석, 초보자 친화적으로 작성.
```

### 🇺🇸 English Prompt
```
Create a Korean Google Colab notebook demonstrating AI Agent Tool Use
with the Anthropic Claude API.

0. Agent terminology glossary: agent, tool, tool schema, agentic loop,
   stop_reason, orchestrator, ReAct pattern — explain with a table and diagram
1. Agent concept: LLM + tools + loop
2. Define 3 tools as Python functions:
   - get_current_time(), calculate(expression), search_company_info(name)
3. Pass tools to Claude using the tools parameter
4. Implement tool use loop using block.input (not block.tool_input):
   while loop: check stop_reason → tool_use → execute → add tool_result → repeat
5. Print agent trace
6. 3 test scenarios
7. Mini exercise: add custom tool
Korean comments, beginner-friendly.
```

# FM2 실습 2: AI 에이전트 — Tool Use(도구 사용)

## 학습 목표
- AI 에이전트의 개념과 핵심 용어를 이해한다
- Anthropic API의 Tool Use 기능을 구현한다
- 에이전트 루프(LLM → 도구 실행 → 결과 반환 → 반복)를 직접 만든다

## 에이전트 vs 챗봇

| | 챗봇 | AI 에이전트 |
|--|------|-------------|
| 동작 | 질문 → 답변 (1회) | 목표 → 계획 → 도구 사용 → 반복 |
| 도구 | 없음 | 검색, 계산, DB조회, API호출 등 |
| 자율성 | 낮음 | 높음 (스스로 판단) |
| 예시 | "파이썬이란?" | "이번 달 매출 보고서 작성해줘" |

## 📖 에이전트 핵심 용어 사전

| 용어 | 쉬운 설명 | 비유 |
|------|----------|------|
| **에이전트 (Agent)** | 목표를 받아 스스로 계획하고 도구를 사용해 완수하는 AI | 비서 |
| **도구 (Tool)** | Claude가 호출할 수 있는 파이썬 함수 | 비서의 업무 도구 (전화기, 계산기) |
| **도구 스키마 (Tool Schema)** | 도구 이름·설명·입력 형식을 JSON으로 정의한 설명서 | 업무 매뉴얼 |
| **에이전트 루프 (Agentic Loop)** | Claude 호출 → 도구 실행 → 결과 전달을 반복하는 구조 | 업무 사이클 |
| **stop_reason** | Claude가 왜 멈췄는지 알려주는 신호 (`end_turn` / `tool_use`) | 업무 상태 보고 |
| **오케스트레이터 (Orchestrator)** | 여러 도구/에이전트를 조율하는 코드 (우리가 만드는 루프) | 팀장 |
| **ReAct 패턴** | **Re**ason(생각) → **Act**(행동) → Observe(관찰) 반복 | 문제 해결 사이클 |

### 에이전트 루프 흐름도
```
사용자 질문
    ↓
Claude에게 전달 (도구 목록 포함)
    ↓
Claude의 응답 확인
    ├── stop_reason = "end_turn"   → ✅ 최종 답변 출력 (종료)
    └── stop_reason = "tool_use"  → 🔧 도구 실행
                                        ↓
                                   결과를 Claude에게 전달
                                        ↓
                                   다시 Claude 호출 (반복)
```

In [ ]:
!pip install anthropic -q
import getpass, anthropic, json, ast
from datetime import datetime

api_key = getpass.getpass("🔑 Anthropic API 키: ")
client = anthropic.Anthropic(api_key=api_key)
print("✅ 준비 완료!")

## 1단계: 도구(Tool) 정의

**도구 = Claude가 호출할 수 있는 파이썬 함수**  
Claude는 언제 어떤 도구를 써야 할지 스스로 판단합니다.

In [ ]:
# ─── 도구 함수 정의 ────────────────────────────────────────────

def get_current_time():
    """현재 날짜와 시간을 반환합니다."""
    return datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분")


def calculate(expression: str):
    """수식을 안전하게 계산합니다. (예: '2024 * 12 + 500')"""
    try:
        # eval 대신 ast를 사용하여 안전하게 수식만 계산
        tree = ast.parse(expression, mode='eval')
        result = eval(compile(tree, '<string>', 'eval'),
                      {"__builtins__": {}}, {})
        return f"{expression} = {result:,}"
    except Exception as e:
        return f"계산 오류: {str(e)}"


def search_company_info(company_name: str):
    """회사 정보를 검색합니다. (가상 데이터)"""
    company_db = {
        "삼성전자": {"업종": "전자/반도체", "직원수": "약 27만명", "2023매출": "258조원", "본사": "수원"},
        "현대자동차": {"업종": "자동차", "직원수": "약 7만명", "2023매출": "162조원", "본사": "서울"},
        "카카오": {"업종": "IT/플랫폼", "직원수": "약 1.2만명", "2023매출": "7.6조원", "본사": "제주"},
        "네이버": {"업종": "IT/포털", "직원수": "약 4천명", "2023매출": "9.7조원", "본사": "성남"},
    }
    info = company_db.get(company_name)
    if info:
        return f"{company_name}: " + ", ".join([f"{k}={v}" for k, v in info.items()])
    return f"'{company_name}' 정보를 찾을 수 없습니다. 등록된 회사: {list(company_db.keys())}"


print("✅ 도구 정의 완료!")
print(f"  - get_current_time() → {get_current_time()}")
print(f"  - calculate('100 * 12') → {calculate('100 * 12')}")
print(f"  - search_company_info('카카오') → {search_company_info('카카오')}")

In [ ]:
# ─── 도구 스키마: Claude에게 도구 사용법을 알려주는 설명서 ──────
# Claude는 이 설명을 보고 언제 어떤 도구를 쓸지 결정합니다
TOOLS = [
    {
        "name": "get_current_time",
        "description": "현재 날짜와 시간을 반환합니다. 날짜나 시간 관련 질문에 사용하세요.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "calculate",
        "description": "수식을 계산합니다. 사칙연산, 백분율 등 수학 계산이 필요할 때 사용하세요.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "계산할 수식 (예: '5000000 * 0.1 + 200000')"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "search_company_info",
        "description": "회사 정보(업종, 직원수, 매출 등)를 검색합니다.",
        "input_schema": {
            "type": "object",
            "properties": {
                "company_name": {
                    "type": "string",
                    "description": "검색할 회사 이름 (예: '삼성전자')"
                }
            },
            "required": ["company_name"]
        }
    }
]

# 도구 이름 → 실제 파이썬 함수 매핑
TOOL_FUNCTIONS = {
    "get_current_time": get_current_time,
    "calculate": calculate,
    "search_company_info": search_company_info,
}

print("✅ 도구 스키마 준비 완료!")

## 2단계: 에이전트 루프 구현

Claude 응답의 `stop_reason`을 보고 계속 반복할지 종료할지 결정합니다.

> **핵심 포인트:** 도구 호출 결과는 `block.input`으로 읽습니다. (`block.tool_input` ❌)

In [ ]:
def run_agent(user_question, verbose=True):
    """에이전트 루프: 도구를 사용하여 질문에 답변합니다."""
    messages = [{"role": "user", "content": user_question}]

    if verbose:
        print(f"\n❓ 질문: {user_question}")
        print("─" * 55)

    step = 0
    while True:
        step += 1
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            tools=TOOLS,       # 사용 가능한 도구 목록 전달
            messages=messages
        )

        # Claude의 전체 응답을 메시지 이력에 추가
        messages.append({"role": "assistant", "content": response.content})

        # ── 케이스 1: 최종 답변 (루프 종료) ──────────────────────
        if response.stop_reason == "end_turn":
            final = next((b.text for b in response.content if hasattr(b, 'text')), "")
            if verbose:
                print(f"💬 최종 답변: {final}")
            return final

        # ── 케이스 2: 도구 사용 요청 ──────────────────────────────
        if response.stop_reason == "tool_use":
            tool_results = []

            for block in response.content:
                if block.type != "tool_use":
                    continue

                tool_name  = block.name
                tool_input = block.input   # ✅ 올바른 속성명: .input
                                           # ❌ 틀린 속성명: .tool_input

                # 실제 파이썬 함수 실행
                func   = TOOL_FUNCTIONS[tool_name]
                result = func(**tool_input) if tool_input else func()

                if verbose:
                    print(f"  [Step {step}] 🔧 도구: {tool_name}")
                    print(f"             📥 입력: {tool_input}")
                    print(f"             📦 결과: {result}")

                tool_results.append({
                    "type":        "tool_result",
                    "tool_use_id": block.id,   # 도구 호출과 결과를 연결하는 ID
                    "content":     str(result)
                })

            # 도구 실행 결과를 Claude에게 전달 → 루프 재시작
            messages.append({"role": "user", "content": tool_results})


print("✅ 에이전트 준비 완료!")

In [ ]:
# 테스트 1: 시간 + 계산 복합 질문
run_agent("지금 몇 월이야? 그리고 월 매출이 4800만원이면 연간 매출은 얼마야?")

In [ ]:
# 테스트 2: 회사 정보 검색 (도구 2회 호출)
run_agent("삼성전자와 네이버의 직원수와 매출을 비교해줘")

In [ ]:
# 테스트 3: 도구가 필요 없는 순수 지식 질문
# → stop_reason이 바로 'end_turn'이 됩니다 (도구 호출 없음)
run_agent("AI 에이전트와 일반 챗봇의 차이점을 2줄로 설명해줘")

## ✏️ 미니 실습: 나만의 도구 추가하기

In [ ]:
# ✏️ 새로운 도구를 정의하고 에이전트에 추가해보세요!

def get_exchange_rate(currency: str):
    """환율 정보를 반환합니다 (가상 데이터)."""
    rates = {"USD": 1350, "JPY": 9.2, "EUR": 1480, "CNY": 186}
    rate = rates.get(currency.upper())
    if rate:
        return f"1 {currency.upper()} = {rate}원"
    return f"{currency} 환율 정보 없음. 지원 통화: {list(rates.keys())}"

# 스키마 정의
TOOLS.append({
    "name": "get_exchange_rate",
    "description": "통화의 현재 환율을 조회합니다. (USD, JPY, EUR, CNY 지원)",
    "input_schema": {
        "type": "object",
        "properties": {
            "currency": {"type": "string", "description": "통화 코드 (예: 'USD', 'JPY')"}
        },
        "required": ["currency"]
    }
})
TOOL_FUNCTIONS["get_exchange_rate"] = get_exchange_rate

# 새 도구 테스트
run_agent("달러 환율이 얼마야? 그리고 500달러는 한국 돈으로 얼마야?")

## 📝 정리

| 개념 | 코드 | 설명 |
|------|------|------|
| 도구 전달 | `tools=TOOLS` | Claude에게 사용 가능한 도구 목록 전달 |
| 도구 호출 감지 | `stop_reason == "tool_use"` | Claude가 도구를 쓰겠다는 신호 |
| 도구 입력 읽기 | `block.input` | 도구에 전달할 인자 딕셔너리 |
| 호출 연결 ID | `block.id` → `tool_use_id` | 도구 호출과 결과를 연결 |
| 종료 감지 | `stop_reason == "end_turn"` | 최종 답변 완성 신호 |

**다음 실습 (FM2-03):** 여러 Claude API 호출을 파이프라인으로 연결하여 주간 보고서를 자동 생성합니다!